In [0]:
# ============================================================
# NOTEBOOK 02 — Silver Transformation
# Medallion Architecture: Bronze Parquet → Silver Parquet
# ============================================================
# Reads Bronze Parquet, cleans and enriches the data, engineers
# features for ML, and saves a clean Silver Parquet layer.
# ============================================================

from pyspark.sql import functions as F

BRONZE_PATH = "/Volumes/workspace/default/flight_raw_data/bronze"
SILVER_PATH = "/Volumes/workspace/default/flight_raw_data/silver"

print("Bronze input path:", BRONZE_PATH)
print("Silver output path:", SILVER_PATH)

Bronze input path: /Volumes/workspace/default/flight_raw_data/bronze
Silver output path: /Volumes/workspace/default/flight_raw_data/silver


In [0]:
# ============================================================
# Step 1 — Read Bronze Parquet files
# ============================================================
# We read from Parquet now instead of CSV. Parquet reads are
# significantly faster because the format is columnar and
# compressed — Spark only reads the columns it needs.
# ============================================================

df_flights = spark.read.parquet(f"{BRONZE_PATH}/flights")
df_airlines = spark.read.parquet(f"{BRONZE_PATH}/airlines")
df_airports = spark.read.parquet(f"{BRONZE_PATH}/airports")

print("flights row count:", df_flights.count())
print("airlines row count:", df_airlines.count())
print("airports row count:", df_airports.count())

flights row count: 5819079
airlines row count: 14
airports row count: 322


In [0]:
# ============================================================
# Step 2 — Filter and clean the flights DataFrame
# ============================================================
# 1. Remove cancelled flights (CANCELLED=1) — they have no
#    delay data so they are useless for delay analysis and ML.
# 2. Drop rows where ARRIVAL_DELAY or DEPARTURE_DELAY is null
#    — these are incomplete records we cannot use.
# 3. Drop rows where ELAPSED_TIME or AIR_TIME is null —
#    these indicate the flight data is fundamentally incomplete.
# ============================================================

df_clean = df_flights.filter(F.col("CANCELLED") == 0)

df_clean = df_clean.dropna(subset=[
    "ARRIVAL_DELAY",
    "DEPARTURE_DELAY",
    "ELAPSED_TIME",
    "AIR_TIME",
    "DISTANCE"
])

print("Rows after removing cancelled and nulls:", df_clean.count())
print("Rows removed:", 5819079 - df_clean.count())

Rows after removing cancelled and nulls: 5714008
Rows removed: 105071


In [0]:
# ============================================================
# Step 3 — Feature Engineering
# ============================================================
# We create two new columns that will be used in ML and analytics:
#
# IS_DELAYED — binary label for ML classification.
#   1 = flight arrived more than 15 minutes late (industry standard
#       definition of a delayed flight used by the FAA and BTS)
#   0 = flight arrived on time or less than 15 minutes late
#
# DELAY_CATEGORY — human-readable bucketing of delay severity
#   On Time   = arrival delay <= 15 minutes
#   Minor     = 15 to 45 minutes late
#   Major     = 45 to 120 minutes late
#   Severe    = more than 120 minutes late
# ============================================================

df_clean = df_clean.withColumn(
    "IS_DELAYED",
    F.when(F.col("ARRIVAL_DELAY") > 15, 1).otherwise(0)
)

df_clean = df_clean.withColumn(
    "DELAY_CATEGORY",
    F.when(F.col("ARRIVAL_DELAY") <= 15, "On Time")
     .when((F.col("ARRIVAL_DELAY") > 15) & (F.col("ARRIVAL_DELAY") <= 45), "Minor")
     .when((F.col("ARRIVAL_DELAY") > 45) & (F.col("ARRIVAL_DELAY") <= 120), "Major")
     .otherwise("Severe")
)

print("Feature engineering complete.")
print("IS_DELAYED distribution:")
df_clean.groupBy("IS_DELAYED").count().show()

print("DELAY_CATEGORY distribution:")
df_clean.groupBy("DELAY_CATEGORY").count().orderBy("count", ascending=False).show()

Feature engineering complete.
IS_DELAYED distribution:
+----------+-------+
|IS_DELAYED|  count|
+----------+-------+
|         0|4690510|
|         1|1023498|
+----------+-------+

DELAY_CATEGORY distribution:
+--------------+-------+
|DELAY_CATEGORY|  count|
+--------------+-------+
|       On Time|4690510|
|         Minor| 586325|
|         Major| 322753|
|        Severe| 114420|
+--------------+-------+



In [0]:
# ============================================================
# Step 4 — Join airlines and airports to enrich flights data
# ============================================================
# Currently flights only has IATA codes like "AA" or "LAX".
# We join airlines to get full airline names.
# We join airports twice — once for origin, once for destination
# — to get city names for both ends of each flight.
# We rename columns before joining to avoid ambiguity.
# ============================================================

# Rename airlines columns before join
df_airlines_renamed = df_airlines.withColumnRenamed("IATA_CODE", "AIRLINE_CODE") \
                                  .withColumnRenamed("AIRLINE", "AIRLINE_NAME")

# Rename airports columns for origin join
df_airports_origin = df_airports.select(
    F.col("IATA_CODE").alias("ORIGIN_CODE"),
    F.col("AIRPORT").alias("ORIGIN_AIRPORT_NAME"),
    F.col("CITY").alias("ORIGIN_CITY"),
    F.col("STATE").alias("ORIGIN_STATE")
)

# Rename airports columns for destination join
df_airports_dest = df_airports.select(
    F.col("IATA_CODE").alias("DEST_CODE"),
    F.col("AIRPORT").alias("DEST_AIRPORT_NAME"),
    F.col("CITY").alias("DEST_CITY"),
    F.col("STATE").alias("DEST_STATE")
)

# Join airlines
df_enriched = df_clean.join(df_airlines_renamed, df_clean["AIRLINE"] == df_airlines_renamed["AIRLINE_CODE"], "left")

# Join origin airport
df_enriched = df_enriched.join(df_airports_origin, df_enriched["ORIGIN_AIRPORT"] == df_airports_origin["ORIGIN_CODE"], "left")

# Join destination airport
df_enriched = df_enriched.join(df_airports_dest, df_enriched["DESTINATION_AIRPORT"] == df_airports_dest["DEST_CODE"], "left")

print("Enriched schema sample:")
df_enriched.select("AIRLINE_NAME", "ORIGIN_CITY", "DEST_CITY", "ARRIVAL_DELAY", "IS_DELAYED", "DELAY_CATEGORY").show(5)

Enriched schema sample:
+--------------------+-------------+---------------+-------------+----------+--------------+
|        AIRLINE_NAME|  ORIGIN_CITY|      DEST_CITY|ARRIVAL_DELAY|IS_DELAYED|DELAY_CATEGORY|
+--------------------+-------------+---------------+-------------+----------+--------------+
|Alaska Airlines Inc.|    Anchorage|        Seattle|          -22|         0|       On Time|
|American Airlines...|  Los Angeles|West Palm Beach|           -9|         0|       On Time|
|     US Airways Inc.|San Francisco|      Charlotte|            5|         0|       On Time|
|American Airlines...|  Los Angeles|          Miami|           -9|         0|       On Time|
|Alaska Airlines Inc.|      Seattle|      Anchorage|          -21|         0|       On Time|
+--------------------+-------------+---------------+-------------+----------+--------------+
only showing top 5 rows


In [0]:
# ============================================================
# Step 5 — Save Silver Parquet
# ============================================================
# We select only the columns we need going forward — dropping
# redundant join key columns like AIRLINE_CODE, ORIGIN_CODE,
# DEST_CODE that were only needed for the join itself.
# ============================================================

df_silver = df_enriched.select(
    "YEAR", "MONTH", "DAY", "DAY_OF_WEEK",
    "AIRLINE", "AIRLINE_NAME",
    "FLIGHT_NUMBER", "TAIL_NUMBER",
    "ORIGIN_AIRPORT", "ORIGIN_AIRPORT_NAME", "ORIGIN_CITY", "ORIGIN_STATE",
    "DESTINATION_AIRPORT", "DEST_AIRPORT_NAME", "DEST_CITY", "DEST_STATE",
    "SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "DEPARTURE_DELAY",
    "SCHEDULED_ARRIVAL", "ARRIVAL_TIME", "ARRIVAL_DELAY",
    "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", "DISTANCE",
    "TAXI_OUT", "TAXI_IN",
    "DIVERTED", "CANCELLED",
    "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY",
    "IS_DELAYED", "DELAY_CATEGORY"
)

print("Saving Silver Parquet...")
df_silver.write.mode("overwrite").parquet(f"{SILVER_PATH}/flights")

print("Silver transformation complete.")
print("Total rows saved:", df_silver.count())
print("Total columns:", len(df_silver.columns))

Saving Silver Parquet...
Silver transformation complete.
Total rows saved: 5714008
Total columns: 37
